# 3 · RAG Query

Ask questions of the transcription database. For each question:

1. the question is embedded with the same embedding model the database was built with (from `pipeline.env`);
2. the most similar passages are retrieved, optionally narrowed by the metadata filters below;
3. Claude (the model in `pipeline.env`) answers from those passages only, citing them as [1], [2], …

Run the cells in order once; after that, change the question (or the filters) and re-run from
**Step 3** onward. The code is in `rag_query.py`.

## Setup

All settings shared by the three notebooks (API keys, Claude model, embedding model, folder names) live in
**`pipeline.env`** in this folder. Open it, fill in your keys, choose the model, save, then run this cell.
If you change `pipeline.env` later, just run this cell again.

In [1]:
from pipeline_common import load_settings
import rag_query

settings = load_settings()
db = rag_query.open_database(settings)

Claude model      : claude-opus-5-5  (effort: high)
Embeddings        : voyage / voyage-3.5
ANTHROPIC_API_KEY : set (...ugAA)
VOYAGE_API_KEY    : set (...Frs4)
Project folder    : /Users/rfreedma/Documents/CRIM_Python/RAG CHROMA MP
Database: chroma-db/transcriptions — 41 chunk(s)


## Step 1 · System prompt and number of passages

The **system prompt** tells Claude how to answer: its role, language, citation style and limits.
**`NUM_RESULTS`** is how many passages are retrieved and shown to Claude. More gives broader answers
but costs more; 5–15 is typical.

In [ ]:
SYSTEM_PROMPT = """You are a research assistant for musicologists working with diplomatic
transcriptions of nineteenth-century manuscript letters and envelopes (mostly German, some French).
Each source is one page or one envelope side; a letter may be spread over several sources.

Answer the question using ONLY the numbered sources provided. Cite every claim with its source
number in square brackets, e.g. [2]. Quote the original when wording matters, and give an English
translation alongside. Keep names, titles and dates exactly as the sources write them.
The transcriptions use editorial markers: [?word?] is an uncertain reading, [illegible] and
[damaged: …] are gaps, ~~text~~ is a deletion, [ins. above: …] an insertion. Say so when an
answer depends on an uncertain reading.
If the sources do not answer the question, say so plainly; do not fill gaps from general knowledge.
Answer in English."""

NUM_RESULTS = 8

## Step 2 · Filters (optional)

Leave a filter as `None` (or `[]`) to ignore it. First, the values available in the database:

In [3]:
rag_query.show_filter_options(db)

27 transcription(s). All metadata fields: anio, anotacion, chunk, chunks_total, coleccion, diario, fecha, fecha_fuente, fecha_num, file_hash, idioma, imagen_fuente, modelo, procesado, recorte, revision, source_file, titular

diario: 'Ahora' (1), 'Crítica' (1), 'El Diario' (1), 'El Pueblo' (2), 'La Acción' (1), 'La Capital' (2), 'La Hora' (1), 'La Libertad' (1), 'La Nación' (5), 'La Prensa' (6), 'La Razón' (1), 'La Verdad' (2), 'Rumbos' (2), 'Símbolo' (1)
anio: 1940 (27)
fecha_fuente: 'manuscrita' (27)
idioma: 'Spanish' (27)
revision: False (5), True (22)
fecha: 1940-07-12 to 1940-11-25


- **`FILTER_SENDER`** / **`FILTER_RECIPIENT`**: lists of names, e.g. `["F. Liszt"]`, spelled exactly as listed above
  (names are recorded as written on each page, so the same person may appear in several spellings).
- **`FILTER_DOCUMENT_TYPE`**: e.g. `["letter"]` or `["envelope"]`.
- **`FILTER_DATE_FROM` / `FILTER_DATE_TO`**: dates as `"1867-03-01"`, `"1867-03"` or `"1867"`.
  Pages without a date (most continuation pages) are left out whenever a date filter is set.
- **`FILTER_NEEDS_REVIEW`**: `True` = only files flagged for review, `False` = only unflagged files.
- **`FILTER_OTHER`**: any other metadata field, e.g. `{"year": 1867}`, `{"place": "Rom"}`, `{"language": "French"}`.
- **`TEXT_MUST_CONTAIN`**: only passages containing this exact text (case-sensitive), e.g. `"Gottschalg"`.

In [ ]:
FILTER_SENDER = []
FILTER_RECIPIENT = []
FILTER_DOCUMENT_TYPE = []
FILTER_DATE_FROM = None
FILTER_DATE_TO = None
FILTER_NEEDS_REVIEW = None
FILTER_OTHER = {}
TEXT_MUST_CONTAIN = None

where = rag_query.build_filter(sender=FILTER_SENDER, recipient=FILTER_RECIPIENT,
                               document_type=FILTER_DOCUMENT_TYPE, date_from=FILTER_DATE_FROM,
                               date_to=FILTER_DATE_TO, needs_review=FILTER_NEEDS_REVIEW,
                               other=FILTER_OTHER)
print("Filter:", where, "| Text must contain:", TEXT_MUST_CONTAIN)

## Step 3 · Ask a question  (uses the embeddings and Claude APIs)

In [ ]:
QUESTION = "What does Liszt write about in these letters, and which people and musical works does he mention?"

result = rag_query.ask(settings, db, QUESTION, system_prompt=SYSTEM_PROMPT, k=NUM_RESULTS,
                       where=where, text_contains=TEXT_MUST_CONTAIN)
rag_query.show_result(result)

## Step 4 · Inspect the sources (optional)

The retrieved passages with their metadata. Lower *distance* = closer match.
To read the passages in full, use `rag_query.show_result(result, full_passages=True)`.

In [6]:
rag_query.sources_table(result)

,#,distance,fecha,chunk,anio,modelo,titular,chunks_total,coleccion,idioma,fecha_fuente,file_hash,procesado,diario,anotacion,imagen_fuente,revision,recorte,fecha_num,source_file
0,1,0.371,1940-07-13,1,1940,claude-opus-5-5,Ofreciéronse anoche novedades en el Colón y Odeón,2,"Ginastera scrapbook, press clippings 1940 (pho...",Spanish,manuscrita,fa553845895c7221,2026-10-03,El Pueblo,"""El Pueblo"" Julio 13 de 1940.",IMG_1950.JPEG,True,1,19400713,IMG_1950_r01.md
1,2,0.421,1940-07-13,2,1940,claude-opus-5-5,"Fué estrenado anoche el ballet ""Panambí"" en el...",3,"Ginastera scrapbook, press clippings 1940 (pho...",Spanish,manuscrita,08d0401c70dafd38,2026-10-03,La Libertad,"""""LA LIBERTAD"" (AVELLANEDA) JULIO 13 DE 1940.""",IMG_1952.JPEG,True,1,19400713,IMG_1952_r01.md
2,3,0.421,1940-08-17,0,1940,claude-opus-5-5,"El autor del ballet ""Panambi"" es un joven veci...",1,"Ginastera scrapbook, press clippings 1940 (pho...",Spanish,manuscrita,2047aa7699e2ae77,2026-10-03,Rumbos,"""RUMBOS"" Agosto 17 de 1940.",IMG_1951.JPEG,True,2,19400817,IMG_1951_r02.md
3,4,0.424,1940-07-13,0,1940,claude-opus-5-5,"Fué estrenado anoche el ballet ""Panambí"" en el...",3,"Ginastera scrapbook, press clippings 1940 (pho...",Spanish,manuscrita,08d0401c70dafd38,2026-10-03,La Libertad,"""""LA LIBERTAD"" (AVELLANEDA) JULIO 13 DE 1940.""",IMG_1952.JPEG,True,1,19400713,IMG_1952_r01.md
4,5,0.426,1940-09-28,0,1940,claude-opus-5-5,Alberto E. Ginastera,2,"Ginastera scrapbook, press clippings 1940 (pho...",Spanish,manuscrita,edb7928d6403625f,2026-10-03,Rumbos,"""RUMBOS"" Septiembre 28 de 1940.",IMG_1951.JPEG,True,1,19400928,IMG_1951_r01.md
5,6,0.454,1940-11-25,0,1940,claude-opus-5-5,"Con gran éxito ofreció anoche un concierto, en...",3,"Ginastera scrapbook, press clippings 1940 (pho...",Spanish,manuscrita,9490ba81cfec8ef0,2026-10-03,La Capital,"""""La Capital"" (Rosario) Noviembre 25 de 1940.""",IMG_1959.JPEG,True,1,19401125,IMG_1959_r01.md
6,7,0.461,1940-07-13,1,1940,claude-opus-5-5,"Fué estrenado anoche el ballet ""Panambí"" en el...",3,"Ginastera scrapbook, press clippings 1940 (pho...",Spanish,manuscrita,08d0401c70dafd38,2026-10-03,La Libertad,"""""LA LIBERTAD"" (AVELLANEDA) JULIO 13 DE 1940.""",IMG_1952.JPEG,True,1,19400713,IMG_1952_r01.md
7,8,0.469,1940-07-12,0,1940,claude-opus-5-5,"Se trata del baile ""Panambi"" de A. E. Ginastera",2,"Ginastera scrapbook, press clippings 1940 (pho...",Spanish,manuscrita,de3537ac5f5f92de,2026-10-03,El Pueblo,"""El Pueblo"" Julio 12 de 1940.",IMG_1950.JPEG,True,2,19400712,IMG_1950_r02.md


## Step 5 · Save (optional)

Saves the question, answer and full source passages as a Markdown file in `rag_answers/`.

In [7]:
rag_query.save_result(result)

Saved to rag_answers/20261003_195342_how_did_the_press_react_to_the_premiere.md


PosixPath('/Users/rfreedma/Documents/CRIM_Python/RAG CHROMA MP/rag_answers/20261003_195342_how_did_the_press_react_to_the_premiere.md')